# Erythroid label transfer

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)
library(Matrix)
library(pheatmap)


<b><font size=5 color=pink >Step 1: map BMO erythroid cells to adult bone marrow</font></b>


In [ ]:
Erythroid <- readRDS(file.path(project_root, "data", "processed", "DBMOs-Erythroid.rds"))
Adult_BM_Erythroid <- readRDS(file.path(project_root, "data", "processed", "Adult_BM_Erythroid.rds"))


In [ ]:
anchors.ABM.Erythroid <- FindTransferAnchors(
  reference = Adult_BM_Erythroid,
  query = Erythroid,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Erythroid <- MapQuery(
  anchorset = anchors.ABM.Erythroid,
  reference = Adult_BM_Erythroid,
  query = Erythroid,
  refdata = list(ABM.Erythroid = "cluster_anno_l2"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Erythroid@meta.data$predicted.ABM.Erythroid)


In [ ]:
DimPlot(Erythroid,
             reduction = "umap",
             group.by = "predicted.ABM.Erythroid",
             label = TRUE,
             repel = TRUE) +
     ggtitle("Projection onto Adult BM Erythroid Atlas")


In [ ]:
DimPlot(Erythroid,
             reduction = "tsne",
             group.by = "predicted.ABM.Erythroid",
             label = TRUE,
             repel = TRUE) +
     ggtitle("Projection onto Adult BM Erythroid Atlas")


In [ ]:
p <- DimPlot(
  Erythroid,
  reduction = "tsne",
  group.by = "predicted.ABM.Erythroid",
  label = TRUE,
  repel = TRUE,
  pt.size = 0.4,
  shuffle = TRUE
) +
  ggtitle("Adult BM erythroid atlas projection") +
  scale_color_manual(values = c(
    "#4E79A7", "#59A14F", "#E15759", "#76B7B2", "#F28E2B",
    "#EDC948", "#B07AA1", "#FF9DA7", "#9C755F", "#BAB0AC"
  )) +
  theme_classic(base_size = 12) +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    axis.title = element_text(face = "bold", size = 11),
    axis.text = element_text(size = 9, colour = "black"),
    legend.position = "right",
    legend.text = element_text(size = 9),
    legend.title = element_blank()
  ) +
  guides(colour = guide_legend(
    override.aes = list(size = 3, alpha = 1)
  ))
p


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(grid)

df <- Embeddings(Erythroid, reduction = "tsne") %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cell")

meta <- Erythroid@meta.data %>%
  tibble::rownames_to_column("cell")

plot_df <- left_join(df, meta, by = "cell")

plot_df$panel_group <- "Male/FA"

# plot_df$panel_group <- factor(plot_df$panel_group, levels = c("Male/FA", "Female/FA"))

cluster_levels <- unique(plot_df$predicted.ABM.Erythroid)
red_palette <- colorRampPalette(c("#FDE0DD", "#FC9272", "#DE2D26", "#67000D"))(length(cluster_levels))

p <- ggplot(plot_df, aes(x = tSNE_1, y = tSNE_2, color = predicted.ABM.Erythroid)) +
  geom_point(size = 0.45, alpha = 0.9, stroke = 0) +
  facet_wrap(~panel_group, ncol = 1, strip.position = "top") +
  scale_color_manual(values = red_palette) +
  coord_equal() +
  theme_classic(base_size = 13) +
  theme(
    strip.background = element_rect(fill = "white", color = "grey40", linewidth = 0.8),
    strip.text = element_text(size = 13, face = "plain", color = "black"),
    panel.border = element_rect(fill = NA, color = "grey40", linewidth = 0.8),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_blank(),
    axis.title = element_blank(),
    legend.position = "none",
    panel.spacing = unit(0.8, "lines"),
    plot.margin = margin(20, 20, 30, 35)
  )

xrange <- range(plot_df$tSNE_1, na.rm = TRUE)
yrange <- range(plot_df$tSNE_2, na.rm = TRUE)

p_final <- p +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(0.32, 0.68), "npc"),
      y = unit(c(0.06, 0.06), "npc"),
      arrow = arrow(length = unit(0.22, "cm"), type = "closed"),
      gp = gpar(lwd = 1.8, col = "black")
    )
  ) +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(0.08, 0.08), "npc"),
      y = unit(c(0.22, 0.62), "npc"),
      arrow = arrow(length = unit(0.22, "cm"), type = "closed"),
      gp = gpar(lwd = 1.8, col = "black")
    )
  ) +
  annotate("text",
           x = xrange[1] + 0.20 * diff(xrange),
           y = yrange[1] - 0.08 * diff(yrange),
           label = "t-SNE 1",
           size = 5) +
  annotate("text",
           x = xrange[1] - 0.08 * diff(xrange),
           y = yrange[1] + 0.35 * diff(yrange),
           label = "t-SNE 2",
           angle = 90,
           size = 5)

p_final

ggsave(
  filename = file.path(project_root, "results", "figures", "Erythroid_tsne_red_style.pdf"),
  plot = p_final,
  width = 4.2,
  height = 6.2,
  units = "in",
  device = cairo_pdf
)


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(grid)

df <- Embeddings(Erythroid, reduction = "tsne") %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cell")

meta <- Erythroid@meta.data %>%
  tibble::rownames_to_column("cell")

plot_df <- left_join(df, meta, by = "cell")
plot_df$panel_group <- "Male/FA"

cluster_levels <- unique(plot_df$predicted.ABM.Erythroid)
red_palette <- colorRampPalette(c("#FDE0DD", "#FC9272", "#DE2D26", "#67000D"))(length(cluster_levels))

p <- ggplot(plot_df, aes(x = tSNE_1, y = tSNE_2, color = predicted.ABM.Erythroid)) +
  geom_point(size = 0.75, alpha = 0.95, stroke = 0) +
  facet_wrap(~panel_group, ncol = 1, strip.position = "top") +
  scale_color_manual(values = red_palette) +
  coord_equal(clip = "off") +
  theme_classic(base_size = 13) +
  theme(
    strip.background = element_rect(fill = "white", color = "grey40", linewidth = 1),
    strip.text = element_text(size = 14, color = "black"),
    panel.border = element_rect(fill = NA, color = "grey40", linewidth = 1),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_blank(),
    axis.title = element_blank(),
    legend.position = "none",
    panel.spacing = unit(0.8, "lines"),
    plot.margin = margin(35, 35, 70, 85)
  )

p_final <- p +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(0.22, 0.74), "npc"),
      y = unit(c(-0.08, -0.08), "npc"),
      arrow = arrow(length = unit(0.25, "cm"), type = "open", ends = "last"),
      gp = gpar(lwd = 2, col = "black")
    )
  ) +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(-0.08, -0.08), "npc"),
      y = unit(c(0.18, 0.62), "npc"),
      arrow = arrow(length = unit(0.25, "cm"), type = "open", ends = "last"),
      gp = gpar(lwd = 2, col = "black")
    )
  ) +
  annotation_custom(
    grob = textGrob(
      "t-SNE 1",
      x = unit(0.36, "npc"),
      y = unit(-0.13, "npc"),
      gp = gpar(fontsize = 16)
    )
  ) +
  annotation_custom(
    grob = textGrob(
      "t-SNE 2",
      x = unit(-0.13, "npc"),
      y = unit(0.40, "npc"),
      rot = 90,
      gp = gpar(fontsize = 16)
    )
  )

p_final

ggsave(
  filename = file.path(project_root, "results", "figures", "Erythroid_tsne_red_style.pdf"),
  plot = p_final,
  width = 4,
  height = 5,
  units = "in",
  device = cairo_pdf
)


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(grid)

df <- Embeddings(Erythroid, reduction = "tsne") %>%
  as.data.frame() %>%
  tibble::rownames_to_column("cell")

meta <- Erythroid@meta.data %>%
  tibble::rownames_to_column("cell")

plot_df <- left_join(df, meta, by = "cell")

plot_df$panel_group <- "Adult BM Erythroid BMO Erythroid"

cluster_levels <- unique(plot_df$predicted.ABM.Erythroid)

red_palette <- colorRampPalette(
  c("#67000D", "#FC9272", "#DE2D26")
)(length(cluster_levels))

names(red_palette) <- cluster_levels

p <- ggplot(
  plot_df,
  aes(
    x = tSNE_1,
    y = tSNE_2,
    color = predicted.ABM.Erythroid
  )
) +
  geom_point(size = 0.75, alpha = 0.95, stroke = 0) +
  facet_wrap(
    ~panel_group,
    ncol = 1,
    strip.position = "top"
  ) +
  scale_color_manual(values = red_palette) +
  coord_equal(clip = "off") +
  theme_classic(base_size = 13) +
  theme(
    strip.background = element_rect(
      fill = "white",
      color = "black",
      linewidth = 0.9
    ),
    strip.text = element_text(
      size = 13,
      color = "black",
      face = "plain"
    ),
    panel.border = element_rect(
      fill = NA,
      color = "black",
      linewidth = 0.9
    ),
    axis.line = element_blank(),
    axis.ticks = element_blank(),
    axis.text = element_blank(),
    axis.title = element_blank(),

    legend.position = "right",
    legend.title = element_blank(),
    legend.text = element_text(size = 10),
    legend.key.size = unit(0.45, "cm"),

    panel.spacing = unit(0.8, "lines"),

    plot.margin = margin(25, 25, 65, 75)
  ) +
  guides(
    color = guide_legend(
      override.aes = list(size = 3, alpha = 1)
    )
  )

p_final <- p +
  annotation_custom(
    grob = textGrob(
      "t-SNE 1",
      x = unit(0.2, "npc"),
      y = unit(-0.08, "npc"),
      gp = gpar(fontsize = 16)
    )
  ) +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(0.35, 0.5), "npc"),
      y = unit(c(-0.08, -0.08), "npc"),
      arrow = arrow(
        length = unit(0.22, "cm"),
        type = "open",
        ends = "last"
      ),
      gp = gpar(lwd = 2, col = "black")
    )
  ) +
  annotation_custom(
    grob = textGrob(
      "t-SNE 2",
      x = unit(-0.08, "npc"),
      y = unit(0.2, "npc"),
      rot = 90,
      gp = gpar(fontsize = 16)
    )
  ) +
  annotation_custom(
    grob = linesGrob(
      x = unit(c(-0.08, -0.08), "npc"),
      y = unit(c(0.35, 0.5), "npc"),
      arrow = arrow(
        length = unit(0.22, "cm"),
        type = "open",
        ends = "last"
      ),
      gp = gpar(lwd = 2, col = "black")
    )
  )

p_final

ggsave(
  filename = file.path(project_root, "results", "figures", "Erythroid_tsne_ABM_mapped_to_BMO_red_style.pdf"),
  plot = p_final,
  width = 5.3,
  height = 5,
  units = "in",
  device = cairo_pdf
)


In [ ]:
prop_df <- Erythroid@meta.data %>%
  dplyr::select(group, predicted.ABM.Erythroid) %>%
  dplyr::filter(!is.na(predicted.ABM.Erythroid)) %>%
  dplyr::mutate(
    group = factor(group, levels = c("Static.25d", "Dynamic.25d", "Dynamic.31d"))
  )

group_n <- prop_df %>% dplyr::count(group, name = "n_group")

stack_df <- prop_df %>%
  dplyr::count(group, predicted.ABM.Erythroid, name = "n") %>%
  dplyr::left_join(group_n, by = "group") %>%
  dplyr::mutate(
    freq = n / n_group,
    pct_label = paste0(round(freq * 100, 1), "%")
  )

head(stack_df, 10)


In [ ]:
cluster_cols <- c("#FC9272", "#DE2D26", "#67000D")


In [ ]:
p_prop <- ggplot(
  stack_df,
  aes(x = group, y = freq, fill = predicted.ABM.Erythroid)
) +
  geom_col(width = 0.65) +

  geom_text(
    aes(label = ifelse(freq >= 0.05, pct_label, "")),
    position = position_stack(vjust = 0.5),
    color = "white",
    size = 3.2,
    fontface = "bold"
  ) +

  scale_y_continuous(
    expand = expansion(mult = c(0, 0.02)),
    labels = scales::percent_format(accuracy = 1)
  ) +
  scale_fill_manual(values = cluster_cols) +
  labs(
    title = "Erythroid: predicted.ABM.Erythroid cluster composition",
    x = NULL,
    y = "Proportion of cells",
    fill = "ABM Erythroid\ncluster"
  ) +
  theme_classic(base_size = 13) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 14
    ),
    legend.position = "right",
    panel.border = element_rect(
      fill = NA,
      color = "black",
      linewidth = 0.8
    ),
    axis.text.x = element_text(
      face = "bold",
      size = 11
    ),
    axis.text.y = element_text(color = "black"),
    axis.title.y = element_text(color = "black"),
    axis.line = element_blank(),
    axis.ticks = element_line(color = "black"),
    panel.grid = element_blank()
  )

p_prop


In [ ]:
library(Cairo)

ggsave(
  filename = file.path(project_root, "results", "figures", "Erythroid_cluster_composition.pdf"),
  plot = p_prop,
  device = cairo_pdf,
  width = 6,
  height = 5,
  units = "in"
)
